# Human-in-the-Loop: ประตูปฏิบัติการล่วงหน้า, การจัดระดับความเสี่ยง, และการบันทึกการตรวจสอบ

README สำหรับบทเรียนนี้แนะนำ Human-in-the-Loop ด้วยตัวอย่างสั้น ๆ ที่ถามผู้ใช้ว่า `APPROVE` หรือ `REJECT` หลังจากที่เอเจนต์ให้คำตอบแล้ว รูปแบบนั้นเป็นจุดเริ่มต้นที่ดี แต่การใช้งาน HITL ในการผลิตจริงมักต้องการสามส่วนเพิ่มเติม:

1. **ประตูปฏิบัติการล่วงหน้า** ที่ทำงาน **ก่อน** ที่เอเจนต์จะดำเนินการขั้นตอนที่มีความเสี่ยง เพื่อให้ค่าใช้จ่าย, ความไม่สามารถย้อนกลับได้ และความหน่วงเวลายังคงอยู่ในระดับที่ควบคุมได้
2. **การจัดระดับความเสี่ยง** เพื่อให้ดำเนินการที่มีความเสี่ยงต่ำโดยอัตโนมัติ, การดำเนินการที่มีความเสี่ยงปานกลางได้รับการอนุมัติเป็นชุด และเฉพาะการดำเนินการที่มีความเสี่ยงสูงเท่านั้นที่ต้องรอมนุษย์
3. **บันทึกการตรวจสอบพร้อมวงจรแก้ไขซ้ำ** เพื่อให้ทุกการตัดสินใจที่ประตูถูกบันทึกเป็น JSONL และการปฏิเสธจะกระตุ้นเอเจนต์ใหม่ด้วยเหตุผลที่มีโครงสร้างแทนที่จะพิมพ์แค่ `Revising...`

โน้ตบุ๊กนี้สร้างแต่ละส่วนเหล่านี้บนพื้นฐานของโครงสร้างเดียวกับ `06-system-message-framework.ipynb` มันทำงานครบวงจรในโหมด `DEMO_MODE = True` (ไม่ต้องการอินพุตโต้ตอบ) หรือด้วยคำสั่งเรียก `input()` จริงเมื่อ `DEMO_MODE = False` หมายเหตุ: ใน DEMO_MODE การพยายามซ้ำของเป้าหมายที่สามถูกสคริปต์ไว้เพื่อให้เห็นกลไกวงจรที่ครบถ้วน การจำแนกซ้ำที่ขับเคลื่อนด้วยการแก้ไขจริงต้องการ `DEMO_MODE = False` และผู้ปฏิบัติงาน

**นอกขอบเขต (จัดการในบทเรียนอื่น):** การรับรองความถูกต้องและการควบคุมการเข้าถึง (ความเสี่ยงข้อ 2 ใน README บทเรียน 06), มิดเดิลแวร์การเรียกใช้เครื่องมือ (บทเรียน 14 การเจาะลึก MAF), รูปแบบการโต้วาทีของหลายเอเจนต์

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## รูปแบบที่ 1: ประตูควบคุมก่อนทำงาน

โค้ดตัวอย่าง HITL ใน README เรียก agent ก่อน แล้วจึงให้ผู้ใช้อนุมัติผลลัพธ์ ซึ่งเป็นการทำงานแบบ **หลังการทำงาน** agent ได้ทำงานไปแล้ว ดังนั้นค่าบริการการเรียก LLM ก็จ่ายไปแล้ว และผลข้างเคียงใด ๆ (เช่น ส่งอีเมล เขียนแถวข้อมูลในฐานข้อมูล โพสต์คอมเมนต์) ก็เกิดขึ้นแล้วเช่นกัน

การทำงานแบบ **ก่อนการทำงาน** จะใส่ประตูควบคุมก่อนที่ agent จะดำเนินการขั้นตอนที่มีความเสี่ยง Agent จะเสนอการกระทำ ประตูควบคุมจะตัดสินใจว่าจะให้ทำงานหรือไม่ และผลข้างเคียงจะเกิดขึ้นเฉพาะเมื่อได้รับการอนุมัติเท่านั้น

| ด้าน | การอนุมัติหลังการทำงาน (ตัวอย่างใน README) | ประตูควบคุมก่อนการทำงาน (โน้ตบุ๊กนี้) |
|---|---|---|
| การทำงานของการอนุมัติเมื่อใด? | หลังจาก agent สร้างผลลัพธ์แล้ว | ก่อนเกิดผลข้างเคียงใด ๆ |
| ค่าใช้จ่าย LLM เมื่อถูกปฏิเสธ | จ่ายไปแล้ว | จ่ายเฉพาะสำหรับข้อเสนอ ไม่ใช่การกระทำ |
| ผลข้างเคียงที่ไม่สามารถย้อนกลับได้ | อาจเกิดขึ้น (การกระทำเกิดขึ้นแล้ว) | ป้องกันได้ |
| ความชัดเจนในการตรวจสอบ | การอนุมัติเป็นการแสดงผลข้อความ | การอนุมัติเป็นบันทึก JSON พร้อมเวลาที่เกิดเหตุ, การกระทำ, เหตุผล |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## รูปแบบที่ 2: การจัดระดับความเสี่ยง

ไม่ใช่ทุกการกระทำที่ต้องได้รับการอนุมัติจากมนุษย์ การตรวจสอบแบบอ่านอย่างเดียวกับ API สาธารณะมีความเสี่ยงแตกต่างจากการส่งอีเมลถึงลูกค้า การปฏิบัติทั้งสองอย่างเหมือนกันจะทำให้เสียความสนใจของผู้ปฏิบัติงานและทำให้ตัวแทนช้าลง

โมเดลง่ายๆ 3 ระดับ:

| ระดับ | ตัวอย่าง | กระบวนการอนุมัติ |
|---|---|---|
| `ต่ำ` (อ่านอย่างเดียว) | ค้นหาฐานความรู้, ตรวจสอบตัวเลือกเที่ยวบิน, ดึงหน้าจากเว็บสาธารณะ | ดำเนินการอัตโนมัติ, บันทึกสำหรับตรวจสอบ |
| `ปานกลาง` (เปลี่ยนแปลงเล็กน้อย) | แคชผลลัพธ์, เพิ่มตัวนับ, กำหนดการแจ้งเตือน | ดำเนินการอัตโนมัติ, แต่ตรวจสอบเป็นชุดรายวัน |
| `สูง` (มีผลต่อต้านภายนอกหรือไม่สามารถย้อนกลับ) | ส่งอีเมล, เก็บเงินจากบัตร, โพสต์ในช่องทางสาธารณะ | รอดำเนินการเมื่อได้รับการอนุมัติจากมนุษย์ |

นี่เป็นการจัดระดับหนึ่ง ระบบในโรงงานมักใช้ระดับที่ละเอียดมากขึ้น (เช่น ระดับสิทธิ์ AWS IAM, ระดับการเข้าถึงตามบทบาท) เวอร์ชัน 3 ระดับด้านล่างเป็นเวอร์ชันที่เล็กที่สุดที่มีประโยชน์สำหรับตัวแทนที่ผสมผสานการอ่านอย่างเดียวและการกระทำที่มีผลข้างเคียง

ตัวจัดประเภทด้านล่างใช้วิธีการประมาณโดยคำหลักเพื่อให้เดโมยังคงแน่นอนและประหยัด ในระบบการผลิต คุณจะเปลี่ยนเป็นตัวจัดประเภทที่เรียนรู้เองหรือเครื่องยนต์นโยบาย

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: บันทึกตรวจสอบและลูปรอบการแก้ไข

`print("Response approved.")` ไม่ใช่บันทึกตรวจสอบ สำหรับความน่าเชื่อถือ การตัดสินใจแต่ละประตูควรถูกบันทึกเป็นเหตุการณ์ที่มีโครงสร้างซึ่งคุณสามารถค้นหา เล่นซ้ำ หรือติดตามตรวจสอบเหตุการณ์ภายหลังได้

สองส่วน:

1. **JSONL แบบเพิ่มอย่างเดียว** หนึ่งบรรทัดต่อการตัดสินใจ พร้อมเวลาประทับ, การกระทำ, ระดับ, การตัดสินใจ, เหตุผล ง่ายต่อการ grep และง่ายต่อการส่งไปยังที่เก็บบันทึกจริงในภายหลัง
2. **ลูปรอบการแก้ไขเมื่อถูกปฏิเสธ** เมื่อประตูส่งกลับ `deny` ตัวแทนจะรีพรอมต์ตัวเองพร้อมเหตุผลการปฏิเสธในบริบท เพื่อให้ข้อเสนอถัดไปสามารถหลีกเลี่ยงปัญหาได้

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## แหล่งข้อมูลเพิ่มเติม

โครงการสาธารณะอื่น ๆ หลายโครงการได้ใช้รูปแบบ HITL เหล่านี้ในรูปแบบต่าง ๆ กัน เปรียบเทียบแนวทางเพื่อค้นหาสิ่งที่เหมาะกับสแตกของคุณ:

- **LangChain** ตัวห่อเครื่องมือ human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): ตัวห่อเครื่องมือที่หยุดการทำงานเพื่อรอป้อนข้อมูลจากมนุษย์
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ได้ปรับโครงสร้างนี้ใหม่): ใช้บทบาทเอเจนต์เฉพาะเพื่อเป็นตัวแทนมนุษย์ในการสนทนาหลายเอเจนต์
- **Semantic Kernel** ตัวกรองฟังก์ชัน ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): มิดเดิลแวร์ที่ทำงานรอบ ๆ การเรียกใช้ฟังก์ชันแต่ละครั้ง เหมาะสำหรับตรรกะการควบคุมประตู

แต่ละโครงการจัดการกับสามรูปแบบย่อยแตกต่างกัน: LangChain ห่อหุ้มเป็นเครื่องมือ, AutoGen ใช้บทบาทเอเจนต์, Semantic Kernel ใช้ตัวกรองมิดเดิลแวร์ อ่านหนึ่งหรือสองการนำไปใช้ให้จบก่อนเลือกการออกแบบสำหรับเอเจนต์ของคุณเอง

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
